# Activity 3: Text Preprocessing with NLTK
**Course:** Natural Language Processing | **Lecture:** 03 - NLP Toolkits and Preprocessing Techniques

**Author:** Muhammad Bilal Hussain
**Portfolio:** [bilalforge.vercel.app](https://bilalforge.vercel.app)

**Objective:** Apply tokenization, cleaning, stop word removal, stemming and lemmatization to a real piece of text and observe how each step changes the data.

**Text source:** Option A - the paragraph provided in the activity sheet.

## Setup

In [1]:
# Import the libraries used throughout the notebook
import re
import string
from collections import Counter

import nltk
import pandas as pd

# Download the NLTK resources required by this activity
nltk.download('punkt_tab')   # sentence and word tokenizer models
nltk.download('stopwords')   # English stop word list
nltk.download('wordnet')     # dictionary used by the lemmatizer
nltk.download('omw-1.4')     # WordNet extra data (needed by some NLTK versions)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## Step 0: Choose the Text Data (Option A)

In [2]:
# The paragraph provided in the activity sheet
text = (
    "Dr. Ahmed's students are running 5 experiments in the lab on 12 March 2025! "
    "They were studying the effects of temperature (25°C and 40°C) on plants. "
    "The results, however, weren't what they expected: the plants grew faster at 25°C. "
    "Should the team repeat the experiments, or should they change the method? "
    "Many researchers believe that repeated testing leads to better results."
)

print(text)
print("\nWord count (split on spaces):", len(text.split()))

Dr. Ahmed's students are running 5 experiments in the lab on 12 March 2025! They were studying the effects of temperature (25°C and 40°C) on plants. The results, however, weren't what they expected: the plants grew faster at 25°C. Should the team repeat the experiments, or should they change the method? Many researchers believe that repeated testing leads to better results.

Word count (split on spaces): 61


## Task 1: Sentence Tokenization

In [3]:
from nltk.tokenize import sent_tokenize

# Split the paragraph into sentences
sentences = sent_tokenize(text)

print("Number of sentences:", len(sentences), "\n")
for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")

Number of sentences: 5 

1. Dr. Ahmed's students are running 5 experiments in the lab on 12 March 2025!
2. They were studying the effects of temperature (25°C and 40°C) on plants.
3. The results, however, weren't what they expected: the plants grew faster at 25°C.
4. Should the team repeat the experiments, or should they change the method?
5. Many researchers believe that repeated testing leads to better results.


**Comment:** The paragraph has 5 sentences and the tokenizer found all 5. The abbreviation **"Dr."** was handled correctly: `sent_tokenize` did not end the sentence after "Dr." because the Punkt model has learned that a period after a known abbreviation is not a sentence boundary.

## Task 2: Word Tokenization

In [4]:
from nltk.tokenize import word_tokenize

# Split the text into word-level tokens
tokens = word_tokenize(text)

print(tokens)
print("\nTotal token count:", len(tokens))

['Dr.', 'Ahmed', "'s", 'students', 'are', 'running', '5', 'experiments', 'in', 'the', 'lab', 'on', '12', 'March', '2025', '!', 'They', 'were', 'studying', 'the', 'effects', 'of', 'temperature', '(', '25°C', 'and', '40°C', ')', 'on', 'plants', '.', 'The', 'results', ',', 'however', ',', 'were', "n't", 'what', 'they', 'expected', ':', 'the', 'plants', 'grew', 'faster', 'at', '25°C', '.', 'Should', 'the', 'team', 'repeat', 'the', 'experiments', ',', 'or', 'should', 'they', 'change', 'the', 'method', '?', 'Many', 'researchers', 'believe', 'that', 'repeated', 'testing', 'leads', 'to', 'better', 'results', '.']

Total token count: 74


**Comment:** `word_tokenize` treats every punctuation mark (`!`, `,`, `(`, `)`, `:`, `?`, `.`) as its own token. Contractions are split into two tokens: **"weren't" becomes `were` + `n't`**, and **"Ahmed's" becomes `Ahmed` + `'s`**. The abbreviation `Dr.` stays as one token. Also, `25°C` is kept together as one token.

## Task 3: Regular Expression Tokenization

In [5]:
from nltk.tokenize import RegexpTokenizer

# 3a. Words that start with a capital letter
cap_tokenizer = RegexpTokenizer(r"[A-Z][\'\w]+")
print("Capitalized words:\n", cap_tokenizer.tokenize(text))

# 3b. All numbers (whole numbers, e.g. 5, 12, 2025, 25, 40)
number_tokenizer = RegexpTokenizer(r"\d+")
print("\nNumbers:\n", number_tokenizer.tokenize(text))

# 3c. Split the text on whitespace (gaps=True means the pattern is the separator)
whitespace_tokenizer = RegexpTokenizer(r"\s+", gaps=True)
print("\nWhitespace split:\n", whitespace_tokenizer.tokenize(text))

Capitalized words:
 ['Dr', "Ahmed's", 'March', 'They', 'The', 'Should', 'Many']

Numbers:
 ['5', '12', '2025', '25', '40', '25']

Whitespace split:
 ['Dr.', "Ahmed's", 'students', 'are', 'running', '5', 'experiments', 'in', 'the', 'lab', 'on', '12', 'March', '2025!', 'They', 'were', 'studying', 'the', 'effects', 'of', 'temperature', '(25°C', 'and', '40°C)', 'on', 'plants.', 'The', 'results,', 'however,', "weren't", 'what', 'they', 'expected:', 'the', 'plants', 'grew', 'faster', 'at', '25°C.', 'Should', 'the', 'team', 'repeat', 'the', 'experiments,', 'or', 'should', 'they', 'change', 'the', 'method?', 'Many', 'researchers', 'believe', 'that', 'repeated', 'testing', 'leads', 'to', 'better', 'results.']


**Comment:** The capital-letter pattern also picks up sentence starters such as *They*, *The* and *Should*, not only proper nouns. "Dr." is returned as `Dr` because the period is not part of `\w`. The whitespace tokenizer keeps punctuation attached to words (e.g. `results,` and `expected:`), which is the main difference from `word_tokenize`.

## Task 4: Cleaning the Text
Steps are applied in this order: lowercase, remove punctuation, remove numbers.

In [6]:
# Step 4.1: Convert the text to lowercase
clean_text = text.lower()
print("After lowercase:\n", clean_text)

After lowercase:
 dr. ahmed's students are running 5 experiments in the lab on 12 march 2025! they were studying the effects of temperature (25°c and 40°c) on plants. the results, however, weren't what they expected: the plants grew faster at 25°c. should the team repeat the experiments, or should they change the method? many researchers believe that repeated testing leads to better results.


In [7]:
# Step 4.2: Remove punctuation (replaced with a space so words do not get glued together)
# The degree sign is not part of string.punctuation, so it is added manually
punctuation = string.punctuation + "°"
clean_text = re.sub(f"[{re.escape(punctuation)}]", " ", clean_text)
print("After removing punctuation:\n", clean_text)

After removing punctuation:
 dr  ahmed s students are running 5 experiments in the lab on 12 march 2025  they were studying the effects of temperature  25 c and 40 c  on plants  the results  however  weren t what they expected  the plants grew faster at 25 c  should the team repeat the experiments  or should they change the method  many researchers believe that repeated testing leads to better results 


In [8]:
# Step 4.3: Remove numbers (and words containing digits)
clean_text = re.sub(r"\w*\d\w*", " ", clean_text)

# Collapse the extra spaces left behind by the removals
clean_text = re.sub(r"\s+", " ", clean_text).strip()
print("After removing numbers:\n", clean_text)

After removing numbers:
 dr ahmed s students are running experiments in the lab on march they were studying the effects of temperature c and c on plants the results however weren t what they expected the plants grew faster at c should the team repeat the experiments or should they change the method many researchers believe that repeated testing leads to better results


**Comment:** After these three steps the text is plain lowercase words. Note that `25°C` became a lone `c` once the number and the degree sign were removed - a small side effect we will see again later.

### Bonus: the same cleaning in a single line using `lambda` and `map`

In [9]:
# One lambda that lowercases, strips punctuation and removes numbers; map applies it to every text in the list
clean_all = lambda t: re.sub(r"\s+", " ", re.sub(r"\w*\d\w*", " ", re.sub(f"[{re.escape(string.punctuation + '°')}]", " ", t.lower()))).strip()

bonus_clean_text = list(map(clean_all, [text]))[0]

print(bonus_clean_text)
print("\nSame result as step-by-step cleaning:", bonus_clean_text == clean_text)

dr ahmed s students are running experiments in the lab on march they were studying the effects of temperature c and c on plants the results however weren t what they expected the plants grew faster at c should the team repeat the experiments or should they change the method many researchers believe that repeated testing leads to better results

Same result as step-by-step cleaning: True


**Comment:** The lambda gives exactly the same result as the three separate steps. Using `map` makes it easy to clean a whole list of texts at once.

## Task 5: Stop Word Removal

In [10]:
# Tokenize the cleaned text (it only contains lowercase words now, so a simple split is enough)
clean_tokens = clean_text.split()
print("Number of cleaned tokens:", len(clean_tokens))

# 5a. Top 5 most frequent words BEFORE removing stop words
freq_before = Counter(clean_tokens)
print("\nTop 5 BEFORE stop word removal:")
for word, count in freq_before.most_common(5):
    print(f"  {word:<12} {count}")

Number of cleaned tokens: 60

Top 5 BEFORE stop word removal:
  the          7
  they         3
  c            3
  experiments  2
  on           2


In [11]:
from nltk.corpus import stopwords

# 5b. Remove English stop words using NLTK's list
stop_words = set(stopwords.words('english'))
filtered_tokens = [word for word in clean_tokens if word not in stop_words]

print("Tokens before:", len(clean_tokens), "| after:", len(filtered_tokens))
print("Removed stop words:", sorted(set(clean_tokens) & stop_words))
print("\nFiltered tokens:\n", filtered_tokens)

# 5c. Top 5 most frequent words AFTER removing stop words
freq_after = Counter(filtered_tokens)
print("\nTop 5 AFTER stop word removal:")
for word, count in freq_after.most_common(5):
    print(f"  {word:<12} {count}")

Tokens before: 60 | after: 33
Removed stop words: ['and', 'are', 'at', 'in', 'of', 'on', 'or', 's', 'should', 't', 'that', 'the', 'they', 'to', 'were', 'weren', 'what']

Filtered tokens:
 ['dr', 'ahmed', 'students', 'running', 'experiments', 'lab', 'march', 'studying', 'effects', 'temperature', 'c', 'c', 'plants', 'results', 'however', 'expected', 'plants', 'grew', 'faster', 'c', 'team', 'repeat', 'experiments', 'change', 'method', 'many', 'researchers', 'believe', 'repeated', 'testing', 'leads', 'better', 'results']

Top 5 AFTER stop word removal:
  c            3
  experiments  2
  plants       2
  results      2
  dr           1


In [12]:
# Side-by-side comparison of both top-5 lists
comparison = pd.DataFrame({
    "Before (word)": [w for w, _ in freq_before.most_common(5)],
    "Before (count)": [c for _, c in freq_before.most_common(5)],
    "After (word)": [w for w, _ in freq_after.most_common(5)],
    "After (count)": [c for _, c in freq_after.most_common(5)],
}, index=range(1, 6))
comparison

,Before (word),Before (count),After (word),After (count)
1,the,7,c,3
2,they,3,experiments,2
3,c,3,plants,2
4,experiments,2,results,2
5,on,2,dr,1


**Comment:** Before removal, the most frequent words are function words such as *the* (7 times) and *they* (3 times). They are frequent but tell us nothing about the topic. After removal, content words like *experiments*, *plants* and *results* (2 times each) show up, and they describe what the paragraph is about. The `c` (3 times) is a leftover from "25°C" / "40°C" after the numbers were removed, so it is a cleaning side effect and not a real word.

## Task 6: Stemming

In [13]:
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer

porter = PorterStemmer()
lancaster = LancasterStemmer()
snowball = SnowballStemmer("english")

# Use each unique token once (order preserved) so the table stays readable
unique_tokens = list(dict.fromkeys(filtered_tokens))

stem_table = pd.DataFrame({
    "Original": unique_tokens,
    "Porter": [porter.stem(w) for w in unique_tokens],
    "Lancaster": [lancaster.stem(w) for w in unique_tokens],
    "Snowball": [snowball.stem(w) for w in unique_tokens],
})
stem_table

,Original,Porter,Lancaster,Snowball
0,dr,dr,dr,dr
1,ahmed,ahm,ahm,ahm
2,students,student,stud,student
3,running,run,run,run
4,experiments,experi,expery,experi
5,lab,lab,lab,lab
6,march,march,march,march
7,studying,studi,study,studi
8,effects,effect,effect,effect
9,temperature,temperatur,temp,temperatur


**Comment:** Lancaster is the most aggressive stemmer and often cuts words very short, while Porter and Snowball give very similar and gentler results. All three produce stems that are not always real words.

## Task 7: Lemmatization

In [14]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# Lemmatize the same tokens; default POS is noun, so a second column treats words as verbs
lemma_table = stem_table.copy()
lemma_table["Lemma (noun)"] = [lemmatizer.lemmatize(w) for w in unique_tokens]
lemma_table["Lemma (verb)"] = [lemmatizer.lemmatize(w, pos="v") for w in unique_tokens]
lemma_table

,Original,Porter,Lancaster,Snowball,Lemma (noun),Lemma (verb)
0,dr,dr,dr,dr,dr,dr
1,ahmed,ahm,ahm,ahm,ahmed,ahmed
2,students,student,stud,student,student,students
3,running,run,run,run,running,run
4,experiments,experi,expery,experi,experiment,experiment
5,lab,lab,lab,lab,lab,lab
6,march,march,march,march,march,march
7,studying,studi,study,studi,studying,study
8,effects,effect,effect,effect,effect,effect
9,temperature,temperatur,temp,temperatur,temperature,temperature


In [15]:
# Show only the rows where the stemmers and the lemmatizer disagree
mask = (lemma_table["Porter"] != lemma_table["Lemma (noun)"]) | (lemma_table["Porter"] != lemma_table["Lemma (verb)"])
lemma_table[mask]

,Original,Porter,Lancaster,Snowball,Lemma (noun),Lemma (verb)
1,ahmed,ahm,ahm,ahm,ahmed,ahmed
2,students,student,stud,student,student,students
3,running,run,run,run,running,run
4,experiments,experi,expery,experi,experiment,experiment
7,studying,studi,study,studi,studying,study
9,temperature,temperatur,temp,temperatur,temperature,temperature
13,however,howev,howev,howev,however,however
14,expected,expect,expect,expect,expected,expect
15,grew,grew,grew,grew,grew,grow
19,change,chang,chang,chang,change,change


**Comment:** Lemmatization returns real dictionary words (for example *experiments -> experiment*, *results -> result*) while stemmers only chop suffixes. With the default noun setting verbs such as *studying* or *running* are left unchanged; passing `pos="v"` fixes them (*studying -> study*, *grew -> grow*). So the lemmatizer is more accurate but needs the part of speech to work at its best.

## Task 8: Reflection
1. **Most frequent term:** Before stop word removal the most frequent term was **"the"** (7 times); after removal the top term was the leftover letter **"c"** (3 times, from "°C"), followed by the meaningful words *experiments*, *plants* and *results* (2 times each).
2. **Stemming vs lemmatization:** Stemming produced non-words such as **"studi"** (from *studying*, Porter and Snowball) and **"experi"** (from *experiments*); other examples are *temperatur*, *howev* and *mani*. The lemmatizer returned real words: *experiment*, *study* (with `pos="v"`), *temperature*, *however* and *many*.
3. **Biggest change:** Stop word removal changed the text the most, cutting the cleaned text from 60 tokens to 33 (about 45% removed), because common function words such as *the*, *they* and *were* make up almost half of the paragraph. Lowercasing and punctuation removal changed the look of the text but not the number of words.
4. **Overall:** Stemming is fast but crude, while lemmatization is more accurate and gives readable output, especially when the part of speech is provided.